# KAS-PPI: thực nghiệm mở rộng lần 2 (đêm 09/10/2026)

Tiếp nối notebook `KAS_PPI_mo_rong_08_10.ipynb`, dùng **cùng thư mục Drive** (`MyDrive/KAS_PPI_mo_rong`), nên các file đã có từ lần trước (`rec_*.json`, `all_kg_rf_f*.json`) được dùng lại. Mọi bước ghi kết quả ra Drive; nếu Colab ngắt kết nối thì Run all lại, bước đã xong sẽ được bỏ qua.

| Mã | Thực nghiệm | Câu hỏi | Thời gian ước tính (4 CPU) |
|---|---|---|---|
| F1 | Đánh giá kiểu ROAD: xoá aspect bằng **điền khuyết** (Gaussian có điều kiện), giữ nguyên mô hình, không huấn luyện lại | Kết luận về fidelity có còn đúng với một toán tử xoá trung lập, không rò rỉ qua mặt nạ? | 30–60 phút |
| F2 | Sanity check (Adebayo 2018): mô hình học trên **nhãn xáo trộn**; RF với **seed khác**; RF so với XGB | Giải thích có thật sự phụ thuộc vào mô hình không, và có ổn định không? | 30–40 phút |
| F3 | **LORE**, bộ giải thích gốc thứ hai của KGsim2vec (thư viện `lore_sa` của nhóm tác giả LORE) | Câu hỏi hội đồng: "sao chưa so với LORE?" | 2–3 giờ (≈ 13 giây/cặp) |
| F4 | KAS với **người chơi mịn hơn** (cây con độ sâu 2 của GO, 1,102 đơn vị), mô hình giữ nguyên | Hạn chế "51 aspect quá thô": KAS có dùng được ở độ mịn cao hơn không? | 20–30 phút |

**Cách chạy qua đêm:** Runtime → Change runtime type → CPU, **High-RAM**. Chạy mục 0, rồi Runtime → Run all.


## 0. Thư mục làm việc và cài đặt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/KAS_PPI_mo_rong'
os.makedirs(WORK, exist_ok=True); os.chdir(WORK); os.makedirs('f09', exist_ok=True)
print('Thư mục làm việc:', os.getcwd(), '| CPU:', os.cpu_count())

In [ ]:
import os
if not os.path.exists('ExplainablePPI'):
    !git clone -q --depth 1 https://github.com/liseda-lab/ExplainablePPI.git
if not os.path.exists('seek'):
    !git clone -q --depth 1 https://github.com/liseda-lab/seek.git
if not os.path.exists('godata/goa_human.gaf'):
    !mkdir -p godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/go-basic.zip -d godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/goa_human.zip -d godata
if not os.path.exists('seek/PPI/Embeddings/Emb_classes_maxdepth4_nwalks100.txt'):
    !cd seek/PPI && unzip -o -q go.zip && unzip -o -q go_annotations.zip && unzip -o -q Embeddings/Emb_classes_maxdepth4_nwalks100.zip -d Embeddings && unzip -o -q Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.zip -d Embeddings
!pip -q install shap xgboost lime lore_sa pyreadr
if os.path.isdir('records'):
    !cp -n records/rec_*.json . 2>/dev/null; true

## Mã nguồn
Mỗi ô ghi một file. Các file của lần trước được ghi lại y nguyên; `plaus2.py` chỉ sửa để nhận cấu hình `kglore`. Các file mới: `lore_kg.py`, `analyze_lore.py`, `road_impute.py`, `analyze_road.py`, `sanity.py`, `analyze_sanity.py`, `kas_fine.py`, `analyze_fine.py`.

In [ ]:
%%writefile kgsim.py
"""Python re-implementation of KGsim2vec features (Resnik-max, IC Seco computed inside each aspect subgraph)."""
import re, math, collections, pickle, os
OBO="http://purl.obolibrary.org/obo/GO_"
def parse_go(owl):
    parents=collections.defaultdict(set); label={}; cur=None
    pat_cls=re.compile(r'<owl:Class rdf:about="http://purl.obolibrary.org/obo/(GO_\d+)">')
    pat_sub=re.compile(r'<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/(GO_\d+)"/>')
    pat_lab=re.compile(r'<rdfs:label[^>]*>([^<]*)</rdfs:label>')
    depth=0
    for line in open(owl,encoding="utf-8"):
        m=pat_cls.search(line)
        if m: cur=m.group(1); continue
        if cur is None: continue
        if "</owl:Class>" in line and line.strip()=="</owl:Class>": cur=None; continue
        m=pat_sub.search(line)
        if m: parents[cur].add(m.group(1)); continue
        m=pat_lab.search(line)
        if m and cur not in label: label[cur]=m.group(1)
    return parents,label
def load_gaf(gaf):
    ann=collections.defaultdict(set)
    for l in open(gaf):
        if l.startswith("!"): continue
        f=l.split("\t")
        if len(f)<5 or "NOT" in f[3]: continue
        ann[f[1]].add("GO_"+f[4].split(":")[1])
    return ann
class KG:
    def __init__(s,owl,gaf,aspects):
        s.parents,s.label=parse_go(owl)
        s.children=collections.defaultdict(set)
        for c,ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        s.ann=load_gaf(gaf)
        s.aspects=aspects
        s._anc={}
        s.sub=[]; s.ic=[]
        for a in aspects:
            members=s.desc(a)|{a}
            nprot=sum(1 for p,ts in s.ann.items() if ts & members)
            N=len(members)+nprot   # SML counts annotated instances as vertices of the reduced graph
            ic={}
            for c in members:
                d=len(s.desc(c))
                ic[c]=1-math.log(d+1)/math.log(N) if N>1 else 1.0
            s.sub.append(members); s.ic.append(ic)
    def desc(s,c,_cache={}):
        if c in _cache: return _cache[c]
        out=set(); st=[c]
        while st:
            x=st.pop()
            for y in s.children.get(x,()):
                if y not in out: out.add(y); st.append(y)
        _cache[c]=out; return out
    def anc(s,c):
        if c in s._anc: return s._anc[c]
        out={c}; st=[c]
        while st:
            x=st.pop()
            for y in s.parents.get(x,()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c]=out; return out
    def closure(s,terms):
        out=set()
        for t in terms: out|=s.anc(t)
        return out
    def features(s,t1,t2):
        """t1,t2: sets of direct annotation terms."""
        A1=s.closure(t1); A2=s.closure(t2)
        common=A1&A2
        v=[]
        for i,a in enumerate(s.aspects):
            m=s.sub[i]
            # protein needs >=1 direct annotation in aspect
            if not (t1 & m) or not (t2 & m): v.append(0.0); continue
            cm=common & m
            v.append(max((s.ic[i][c] for c in cm),default=0.0))
        return v


In [ ]:
%%writefile seekrep.py
"""Python re-implementation of SEEK pair representations (Sousa et al., KR 2023):
pair vector = average RDF2Vec embedding of the disjoint common ancestors (DCA) of the two proteins' GO annotations.
Class embeddings are the authors' precomputed ones (fixed); only the DCA set changes when annotations are removed."""
import re, json, collections, numpy as np
from kgsim import parse_go
class SEEK:
    def __init__(s, owl, gaf, emb_file):
        s.parents, s.label = parse_go(owl)
        s.children = collections.defaultdict(set)
        for c, ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        # SEEK's build_kg reads every GAF line (qualifier column ignored), protein id = column 2
        s.ann = collections.defaultdict(set)
        with open(gaf) as f:
            f.readline()
            for l in f:
                if l.startswith("!"): continue
                c = l.split("\t")
                s.ann[c[1]].add("GO_" + c[4].split(":")[1])
        txt = open(emb_file).read().replace("'", '"')
        raw = json.loads(re.sub(r'"http://purl.obolibrary.org/obo/', '"', txt))
        s.emb = {k: np.asarray(v, dtype=np.float64) for k, v in raw.items()}
        s.dim = len(next(iter(s.emb.values())))
        s._anc = {}
    def anc(s, c):
        if c in s._anc: return s._anc[c]
        out = {c}; st = [c]
        while st:
            x = st.pop()
            for y in s.parents.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c] = frozenset(out); return s._anc[c]
    def desc(s, c, _cache={}):
        if c in _cache: return _cache[c]
        out = set(); st = [c]
        while st:
            x = st.pop()
            for y in s.children.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        _cache[c] = out; return out
    def dca(s, t1, t2):
        A1 = set().union(*[s.anc(t) for t in t1]) if t1 else set()
        A2 = set().union(*[s.anc(t) for t in t2]) if t2 else set()
        common = A1 & A2
        strict = set()
        for d in common: strict |= (s.anc(d) - {d})
        return [c for c in common - strict if c in s.emb]
    def rep(s, t1, t2):
        d = s.dca(t1, t2)
        if not d: return np.zeros(s.dim)
        return np.mean([s.emb[c] for c in d], 0)


In [ ]:
%%writefile build_feats.py
import sys,time,numpy as np,pickle
sys.path.insert(0,".")
from kgsim import KG
R="ExplainablePPI/Experimental_Results/"
aspects=[l.split("\t")[0].split("/")[-1] for l in open(R+"Results/kgsim2vec_sas.txt") if l.strip()]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects)
rows=[l.rstrip("\n").split("\t") for l in open(R+"Results/ss_resnik_max_ICseco.txt")]
lab={}
for l in open(R+"Data/v11(score950).txt"):
    f=l.split(); lab[(f[0],f[1])]=int(f[2])
keys=[(r[0].replace("http://",""),r[1].replace("http://","")) for r in rows]
Xref=np.array([[float(x) for x in r[2:]] for r in rows])
t=time.time()
Xpy=np.array([kg.features(kg.ann[a],kg.ann[b]) for a,b in keys])
print("built",time.time()-t)
y=np.array([lab[k] for k in keys])
np.savez("feats.npz",Xref=Xref,Xpy=Xpy,y=y,keys=np.array(keys),aspects=np.array(aspects))
d=np.abs(Xref-Xpy)
print("exact pairs",(d.max(1)<1e-6).mean(),"cells within 1e-6",(d<1e-6).mean(),"within 0.05",(d<0.05).mean())
print("zero-pattern agreement",((Xref>0)==(Xpy>0)).mean())
cors=[np.corrcoef(Xref[:,j],Xpy[:,j])[0,1] for j in range(52) if Xref[:,j].std()>0 and Xpy[:,j].std()>0]
print("per-aspect pearson median %.4f min %.4f"%(np.median(cors),np.min(cors)))


In [ ]:
%%writefile build_seek_feats.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
K=np.load("feats_kg.npz"); aspects=list(K["aspects"]); keys=[tuple(k) for k in K["keys"]]
roots=["GO_0008150","GO_0003674","GO_0005575"]
ch=set().union(*[sk.children[r] for r in roots])
print("aspects in SEEK GO:",len(ch),"overlap with KG aspects",len(ch&set(aspects)),"missing",set(aspects)-ch)
cx=sk.desc("GO_0032991")|{"GO_0032991"}
Xs=np.zeros((len(keys),sk.dim)); Xs51=np.zeros_like(Xs); t=time.time()
for i,(a,b) in enumerate(keys):
    t1,t2=sk.ann[a],sk.ann[b]
    Xs[i]=sk.rep(t1,t2); Xs51[i]=sk.rep(t1-cx,t2-cx)
print("built",time.time()-t, "rows changed by complex removal",(np.abs(Xs-Xs51).max(1)>0).mean())
np.savez("feats_seek.npz",Xseek=Xs,Xseek51=Xs51)


In [ ]:
%%writefile explain_v2.py
"""Direction D, full basic experiment: knowledge-space aspect Shapley (D) on three pipelines.
usage: python3 explain_v2.py PIPE MODEL FOLD N    PIPE in {kg,seek,hyb}, MODEL in {rf,xgb}
Players = GO aspects (51 direct children of the 3 GO roots, 'protein-containing complex' removed) in which
both proteins have >=1 direct annotation. Removing a player = deleting both proteins' annotations under that
aspect and recomputing the features through the pipeline (KGsim2vec: Resnik-max/IC-Seco; SEEK: DCA + mean
RDF2Vec embedding with the authors' fixed class embeddings)."""
import sys, time, json, itertools, math, random
import numpy as np
sys.path.insert(0,"."); sys.path.insert(0,"ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap
from lime import lime_tabular

PIPE,MODEL,FOLD,N_PAIRS=sys.argv[1],sys.argv[2],int(sys.argv[3]),int(sys.argv[4])
M_PERM=100; EXACT_MAX=8; TOPK=5
rng=np.random.default_rng(FOLD); random.seed(FOLD)

K=np.load("feats_kg.npz"); y=K["y"]; keys=[tuple(k) for k in K["keys"]]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]; A=[aspects[j] for j in keep]
SF=np.load("feats_seek.npz")
X={"kg":K["Xpy"][:,keep],"seek":SF["Xseek51"],"hyb":np.hstack([K["Xpy"][:,keep],SF["Xseek51"]])}[PIPE]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects); sub_kg=[kg.sub[j] for j in keep]
S_="seek/PPI/"
sk=SEEK(S_+"go.owl",S_+"go_annotations.gaf",S_+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
CX=sk.desc("GO_0032991")|{"GO_0032991"}
sub_sk=[sk.desc(a)|{a} for a in A]

P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
test=[int(l) for l in open(P+("1(complete)" if FOLD==1 else str(FOLD))+".txt") if l.strip()]
m=np.zeros(len(y),bool); m[test]=True
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
clf=(RandomForestClassifier(100,n_jobs=1,random_state=0) if MODEL=="rf"
     else XGBClassifier(n_estimators=300,n_jobs=1,random_state=0,eval_metric="logloss") if MODEL=="xgb"
     else make_pipeline(StandardScaler(),MLPClassifier((100,),max_iter=300,early_stopping=True,random_state=0))).fit(X[~m],y[~m])
def proba(Z): return clf.predict_proba(np.atleast_2d(Z))[:,1].astype(np.float64)

# state = (kg annotations p1, kg annotations p2, seek annotations p1, seek annotations p2)
def feats(st):
    k1,k2,s1,s2=st; parts=[]
    if PIPE in("kg","hyb"):
        v=kg.features(k1,k2); parts.append([v[j] for j in keep])
    if PIPE in("seek","hyb"): parts.append(list(sk.rep(s1,s2)))
    return np.concatenate(parts)
def players_of(st):
    k1,k2,s1,s2=st; out=[]
    for i in range(len(A)):
        a=(PIPE!="seek") and bool(k1&sub_kg[i]) and bool(k2&sub_kg[i])
        b=(PIPE!="kg") and bool(s1&sub_sk[i]) and bool(s2&sub_sk[i])
        if a or b: out.append(i)
    return out
def remove(st,S):
    if not S: return st
    rk=set().union(*[sub_kg[i] for i in S]); rs=set().union(*[sub_sk[i] for i in S])
    k1,k2,s1,s2=st; return (k1-rk,k2-rk,s1-rs,s2-rs)

def shapley(value_batch, players, seed):
    k=len(players); phi=np.zeros(len(A))
    if k==0: return phi
    if k<=EXACT_MAX:
        subsets=[frozenset(c) for r in range(k+1) for c in itertools.combinations(players,r)]
        vals=dict(zip(subsets,value_batch(subsets)))
        for i in players:
            others=[p for p in players if p!=i]
            for r in range(k):
                w=math.factorial(r)*math.factorial(k-r-1)/math.factorial(k)
                for c in itertools.combinations(others,r):
                    S=frozenset(c); phi[i]+=w*(vals[S]-vals[S|{i}])
        return phi
    r=random.Random(seed); perms=[r.sample(players,k) for _ in range(M_PERM)]
    subsets=set()
    for p in perms:
        for t in range(k+1): subsets.add(frozenset(p[:t]))
    subsets=list(subsets); vals=dict(zip(subsets,value_batch(subsets)))
    for p in perms:
        for t,i in enumerate(p): phi[i]+=vals[frozenset(p[:t])]-vals[frozenset(p[:t+1])]
    return phi/M_PERM

def explain_D(st,seed=0):
    pl=players_of(st)
    return shapley(lambda subs: proba(np.array([feats(remove(st,S)) for S in subs])),pl,seed)
def explain_LOO(st,seed=0):   # leave-one-aspect-out occlusion in knowledge space (aspect-level analogue of SEEK 'necessary')
    pl=players_of(st); phi=np.zeros(len(A))
    if not pl: return phi
    v=proba(np.array([feats(st)]+[feats(remove(st,[i])) for i in pl]))
    for j,i in enumerate(pl): phi[i]=v[0]-v[j+1]
    return phi
def explain_FS0(st,seed=0):   # KGsim2vec only: same players and sampler, feature set to 0 instead of recomputed
    pl=players_of(st); x=feats(st)
    def vb(subs):
        Z=np.repeat(x[None],len(subs),0)
        for r,S in enumerate(subs): Z[r,list(S)]=0
        return proba(Z)
    return shapley(vb,pl,seed)
bg=X[~m][rng.choice((~m).sum(),100,replace=False)]
tree=shap.TreeExplainer(clf,data=bg,feature_perturbation="interventional") if MODEL!="mlp" else None
def explain_TS(st,seed=0):    # kg: per aspect feature; hyb: only the 51 named KGsim2vec columns map to aspects
    x=feats(st); sv=tree.shap_values(x[None],check_additivity=False)
    sv=sv[1] if isinstance(sv,list) else sv
    sv=np.array(sv); sv=sv[0,:,1] if sv.ndim==3 else sv[0]
    return np.asarray(sv[:len(A)],dtype=np.float64)
if PIPE=="kg":
    lime_exp=lime_tabular.LimeTabularExplainer(X[~m][:2000],feature_names=A,class_names=["0","1"],discretize_continuous=True,random_state=0)
def explain_LIME(st,seed=0):
    x=feats(st); lime_exp.random_state=np.random.RandomState(seed)
    e=lime_exp.explain_instance(x,lambda Z: clf.predict_proba(Z).astype(np.float64),num_features=10,num_samples=2000,labels=(1,))
    phi=np.zeros(len(A))
    for j,w in e.as_map()[1]: phi[j]=w
    return phi
METHODS={"D":explain_D,"LOO":explain_LOO}
if PIPE=="kg": METHODS.update({"FS0":explain_FS0,"TreeSHAP":explain_TS,"LIME":explain_LIME})
if PIPE=="hyb": METHODS.update({"TreeSHAP_KGpart":explain_TS})
if MODEL=="mlp": METHODS={k:v for k,v in METHODS.items() if not k.startswith("TreeSHAP")}
DETERMINISTIC={"LOO","TreeSHAP","TreeSHAP_KGpart"}

def ranking(phi,pred):
    s=phi if pred==1 else -phi
    return [int(j) for j in np.argsort(-s) if s[j]>0]
def deletion_ks(st,order,pred):
    Z=np.array([feats(remove(st,order[:k])) for k in range(1,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def insertion_ks(st,order,pred,pl):  # start with every player removed, add back the top-k
    Z=np.array([feats(remove(st,[i for i in pl if i not in order[:k]])) for k in range(0,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def eff_size(phi):
    a=np.sort(np.abs(phi))[::-1]; tot=a.sum()
    return int(np.searchsorted(np.cumsum(a)/tot,0.8)+1) if tot>0 else 0
def jacc(a,b,k=3):
    A_,B_=set(a[:k]),set(b[:k]); return len(A_&B_)/len(A_|B_) if (A_|B_) else 1.0

pos=[i for i in test if y[i]==1]; neg=[i for i in test if y[i]==0]
idx=list(rng.choice(pos,N_PAIRS//2,replace=False))+list(rng.choice(neg,N_PAIRS//2,replace=False))
rec=[]; t0=time.time()
for n,i in enumerate(idx):
    p1,p2=keys[i]
    st=(set(kg.ann[p1]),set(kg.ann[p2]),set(sk.ann[p1])-CX,set(sk.ann[p2])-CX)
    p=proba(feats(st))[0]; pred=int(p>0.5); conf=p if pred else 1-p
    pl=players_of(st)
    r=dict(i=int(i),pair=f"{p1}-{p2}",y=int(y[i]),pred=pred,conf=float(conf),k=len(pl))
    # continuity: drop ONE random direct annotation (same GO term in both annotation sets) lying in a player aspect
    cand=sorted({(w,t) for w in (0,1) for t in (st[w]|st[w+2]) if any(t in sub_kg[a] or t in sub_sk[a] for a in pl)})
    pert=None
    if cand:
        w,t=cand[rng.integers(len(cand))]; q=list(st); q[w]=q[w]-{t}; q[w+2]=q[w+2]-{t}; q=tuple(q)
        if int(proba(feats(q))[0]>0.5)==pred: pert=q
    ro=[int(v) for v in rng.permutation(pl)] if pl else []
    r["del_random"]=deletion_ks(st,ro,pred) if ro else None
    r["ins_random"]=insertion_ks(st,ro,pred,pl) if ro else None
    for name,f in METHODS.items():
        ts_=time.time(); phi=f(st,0); r[f"time_{name}"]=time.time()-ts_
        order=ranking(phi,pred)
        r[f"cov_{name}"]=int(len(order)>0); r[f"size_{name}"]=eff_size(phi)
        r[f"del_{name}"]=deletion_ks(st,order,pred) if order else None
        r[f"ins_{name}"]=insertion_ks(st,order,pred,pl) if order else None
        r[f"top3_{name}"]=[A[j] for j in order[:3]]
        r[f"phi_{name}"]={A[j]:round(float(phi[j]),5) for j in np.nonzero(phi)[0]}
        if name in DETERMINISTIC or (name=="D" and len(pl)<=EXACT_MAX):
            r[f"cons_jacc_{name}"]=1.0; r[f"cons_rho_{name}"]=1.0
        else:
            phi2=f(st,1); o2=ranking(phi2,pred); r[f"cons_jacc_{name}"]=jacc(order,o2)
            r[f"cons_rho_{name}"]=float(spearmanr(phi,phi2).correlation) if np.std(phi)>0 and np.std(phi2)>0 else None
        r[f"cont_{name}"]=jacc(order,ranking(f(pert,0),pred)) if pert is not None else None
    rec.append(r)
    if n%25==0: print(PIPE,MODEL,FOLD,n,round(time.time()-t0,1),"s k=",len(pl),flush=True)
json.dump(dict(pipe=PIPE,model=MODEL,fold=FOLD,aspects=A,labels={a:kg.label.get(a,a) for a in A},records=rec),open(f"rec_{PIPE}_{MODEL}_f{FOLD}_{N_PAIRS}.json","w"))
print("done",time.time()-t0)


In [ ]:
%%writefile explain_all.py
"""Explain EVERY test pair of one fold (KGsim2vec view) with KAS, LOO, FS0, TreeSHAP, LIME, for full-data ROAR.
usage: python3 explain_all.py MODEL FOLD [PART NPARTS]   -> all_kg_{MODEL}_f{FOLD}[_p{PART}].json
(each pair explained by the fold model that did not train on it; PART/NPARTS split a fold across CPUs)"""
import sys, json, time
import numpy as np
MODEL, FOLD = sys.argv[1], int(sys.argv[2])
PART, NPARTS = (int(sys.argv[3]), int(sys.argv[4])) if len(sys.argv) > 4 else (0, 1)
sys.argv = ["x", "kg", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])   # setup: data, model, feats, explainers
out = []; t0 = time.time()
for n, i in enumerate(test[PART::NPARTS]):
    p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(), set())
    p = proba(feats(st))[0]; r = dict(i=int(i), y=int(y[i]), pred=int(p > 0.5))
    for name, f in METHODS.items():
        phi = f(st, 0); r[f"phi_{name}"] = {A[j]: round(float(phi[j]), 5) for j in np.nonzero(phi)[0]}
    out.append(r)
    if n % 500 == 0: print(FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, aspects=A, records=out), open(f"all_kg_{MODEL}_f{FOLD}" + (f"_p{PART}" if NPARTS > 1 else "") + ".json", "w"))
print("done", FOLD, time.time() - t0)


In [ ]:
%%writefile get_msigdb.py
"""Extract Reactome and KEGG gene sets (gene symbols) from MSigDB 7.5.1 as bundled in the conda-forge package r-msigdbr 7.5.1
(R lazy-load database, LZMA-compressed objects). Writes REACTOME_msigdb751.gmt and KEGG_msigdb751.gmt. Needs pyreadr."""
import os, io, lzma, struct, tarfile, urllib.request, tempfile, pyreadr
URL = "https://conda.anaconda.org/conda-forge/noarch/r-msigdbr-7.5.1-r42hc72bb7e_1.tar.bz2"
tmp = tempfile.mkdtemp()
tb = os.path.join(tmp, "m.tar.bz2"); urllib.request.urlretrieve(URL, tb)
with tarfile.open(tb) as t:
    b = t.extractfile("lib/R/library/msigdbr/R/sysdata.rdb").read()
objs = []; off = 0
while off < len(b):
    dec = lzma.LZMADecompressor(format=lzma.FORMAT_RAW, filters=[{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}])
    d = dec.decompress(b[off + 5:]); used = len(b) - off - 5 - len(dec.unused_data)
    p = os.path.join(tmp, f"o{len(objs)}.rds"); open(p, "wb").write(d); objs.append(pyreadr.read_r(p)[None]); off += 5 + used
genes, gsg, gs = objs  # msigdbr_genes, msigdbr_geneset_genes, msigdbr_genesets
for sub in ["CP:REACTOME", "CP:KEGG"]:
    s = gs[gs.gs_subcat == sub]; m = gsg[gsg.gs_id.isin(s.gs_id)].merge(genes, on="gene_id"); name = s.set_index("gs_id")["gs_name"]
    with open(sub.split(":")[1] + "_msigdb751.gmt", "w") as f:
        for gid, grp in m.groupby("gs_id"):
            f.write(name[gid] + "\t" + gid + "\t" + "\t".join(sorted(set(grp.human_gene_symbol))) + "\n")
    print(sub, len(s), "gene sets")


In [ ]:
%%writefile seeknat.py
"""SEEK's own explainer (Sousa et al. 2023, generate_seek_explanation.py), made graded and mapped to the 51 fixed aspects.
SEEK explains in DCA space: for each disjoint common ancestor c of the pair it asks the model
  'without c'  (average embedding of the other DCAs; zero vector if none)  -> c is NECESSARY if the predicted class flips
  'only c'     (embedding of c alone)                                        -> c is SUFFICIENT if the predicted class stays.
Graded versions used here (toward interaction, i.e. class-1 probability):
  SEEKN(c) = p(all DCAs) - p(without c)      SEEKS(c) = p(only c) - 0.5
Aspect score: SEEKN(a) = sum of SEEKN(c) over the DCAs c lying in aspect a's subtree;
              SEEKS(a) = SEEKS(c*) for the DCA c* in a that most supports the predicted class.
Each DCA of a pair lies only in player aspects (both proteins have descendants of c), so the mapping never leaves the players.
DCAs outside every aspect (e.g. the BP/MF/CC roots) are counted but cannot be attributed."""
import numpy as np

def dca_vec(sk, D):
    return np.mean([sk.emb[c] for c in D], 0) if D else np.zeros(sk.dim)

def native(st, sk, sub_sk, proba, nA):
    s1, s2 = st[2], st[3]
    D = sk.dca(s1, s2)
    Z = [dca_vec(sk, D)] + [dca_vec(sk, [d for d in D if d != c]) for c in D] + [sk.emb[c] for c in D]
    p = proba(np.array(Z)); p0 = p[0]; m = len(D)
    pw, po = p[1:1 + m], p[1 + m:]
    pred = int(p0 > 0.5); sgn = 1 if pred else -1
    nec = [c for c, q in zip(D, pw) if int(q > 0.5) != pred]
    suf = [c for c, q in zip(D, po) if int(q > 0.5) == pred]
    asp_of = {c: [a for a in range(nA) if c in sub_sk[a]] for c in D}
    phiN = np.zeros(nA); phiS = np.zeros(nA); best = np.full(nA, -np.inf)
    for c, q_w, q_o in zip(D, pw, po):
        for a in asp_of[c]:
            phiN[a] += p0 - q_w
            v = q_o - 0.5
            if sgn * v > best[a]: best[a] = sgn * v; phiS[a] = v
    stats = dict(n_dca=m, n_dca_noasp=sum(1 for c in D if not asp_of[c]), n_nec=len(nec), n_suf=len(suf),
                 nec_asp=sorted({a for c in nec for a in asp_of[c]}), suf_asp=sorted({a for c in suf for a in asp_of[c]}))
    return phiN, phiS, stats, D, asp_of

def dca_curves(D, asp_of, order, pl, pred, sk, proba, topk=5):
    """Deletion / insertion in SEEK's own (DCA) space: drop the DCAs that lie in the removed aspects and average the
    rest, without recomputing annotations. Returns predicted-class probabilities (deletion k=1..topk, insertion k=0..topk)."""
    def keep_without(S):
        S = set(S); return [c for c in D if not (set(asp_of[c]) & S)]
    Zd = [dca_vec(sk, keep_without(order[:k])) for k in range(1, topk + 1)]
    Zi = [dca_vec(sk, keep_without([a for a in pl if a not in order[:k]])) for k in range(0, topk + 1)]
    p = proba(np.array(Zd + Zi)); p = p if pred == 1 else 1 - p
    return list(p[:topk]), list(p[topk:])


In [ ]:
%%writefile seek_native.py
"""E1: SEEK's own explainer vs KAS on the SEEK view, same 2,200 pairs per model as the main runs (rec_seek_{MODEL}_f*.json).
usage: python3 seek_native.py MODEL FOLD      -> rec_seekx_{MODEL}_f{FOLD}.json
Two removal operators: 'ks' = delete annotations + recompute (KAS's value function, same code as explain_v2),
'dca' = drop DCA vectors of the removed aspects, no recompute (SEEK's own operator)."""
import sys, json, glob, time
import numpy as np
MODEL, FOLD = sys.argv[1], int(sys.argv[2])
sys.argv = ["x", "seek", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
from seeknat import native, dca_curves
recs = []
for fn in glob.glob(f"rec_seek_{MODEL}_f{FOLD}_*.json"): recs += json.load(open(fn))["records"]
out = []; t0 = time.time()
for n, r in enumerate(recs):
    i = r["i"]; p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(sk.ann[p1]) - CX, set(sk.ann[p2]) - CX)
    pred = r["pred"]; pl = players_of(st)
    ts = time.time(); phiN, phiS, stats, D, asp_of = native(st, sk, sub_sk, proba, len(A)); tn = time.time() - ts
    phis = {"D": np.array([r["phi_D"].get(a, 0.0) for a in A]), "LOO": np.array([r["phi_LOO"].get(a, 0.0) for a in A]),
            "SEEKN": phiN, "SEEKS": phiS}
    o = dict(i=i, pair=r["pair"], y=r["y"], pred=pred, k=len(pl), time_SEEK=tn, **stats)
    rr = np.random.default_rng(i); orders = {m: ranking(f, pred) for m, f in phis.items()}
    orders["random"] = [int(v) for v in rr.permutation(pl)]
    for m, f in phis.items():
        o[f"phi_{m}"] = {A[j]: round(float(f[j]), 5) for j in np.nonzero(f)[0]}
        o[f"size_{m}"] = eff_size(f); o[f"cov_{m}"] = int(len(orders[m]) > 0)
    for m, od in orders.items():
        if not od: continue
        if m in ("D", "LOO"): o[f"del_ks_{m}"], o[f"ins_ks_{m}"] = r[f"del_{m}"], r[f"ins_{m}"]
        else: o[f"del_ks_{m}"], o[f"ins_ks_{m}"] = deletion_ks(st, od, pred), insertion_ks(st, od, pred, pl)
        o[f"del_dca_{m}"], o[f"ins_dca_{m}"] = dca_curves(D, asp_of, od, pl, pred, sk, proba)
    o["top1_D"] = A[orders["D"][0]] if orders["D"] else None
    for m in ("LOO", "SEEKN", "SEEKS"):
        a, b = orders["D"][:3], orders[m][:3]
        o[f"jac3_D_{m}"] = len(set(a) & set(b)) / len(set(a) | set(b)) if (a or b) else 1.0
        o[f"top1same_D_{m}"] = int(bool(a) and bool(b) and a[0] == b[0])
    out.append({k: (float(v) if isinstance(v, np.floating) else v) for k, v in o.items()})
    if n % 100 == 0: print(MODEL, FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(pipe="seekx", model=MODEL, fold=FOLD, aspects=A, records=out), open(f"rec_seekx_{MODEL}_f{FOLD}.json", "w"))
print("done", MODEL, FOLD, round(time.time() - t0))


In [ ]:
%%writefile plaus2.py
"""Quantitative Plausibility proxy: does an explanation's top aspect agree with external pathway knowledge?
usage: python3 plaus2.py GMT_NAME [CFGS e.g. seekx|rf,seekx|xgb] [TAG]     (reads {GMT_NAME}_msigdb751.gmt: REACTOME / KEGG from MSigDB 7.5.1, or REACTOME_LIVE
       written by get_reactome_live.py from the current reactome.org UniProt2Reactome file)
For every explained pair with y=1 and pred=1 (all rec_*_{rf,xgb}_f*.json), take the smallest gene set (10..300 members in the
GAF universe) that contains both proteins. Reference relevance of an aspect = over-representation (one-sided hypergeometric,
BH over the 51 aspects) of the OTHER members of that pathway (the two proteins excluded) among proteins annotated under the
aspect, versus all GAF proteins. Annotations used for the reference exclude evidence IEA and anything assigned_by Reactome,
so the reference does not reuse Reactome-derived GO annotations.
A pair is informative when it has >=2 players and at least one relevant and one non-relevant player.
Metrics over the pair's players: hit@1, precision@3, NDCG@3 (gain = -log10 p, capped at 10).
Baselines: random order (exact expectation for hit@1, 200 permutations otherwise) and, for KGsim2vec, ranking players by
their raw similarity feature (model-free)."""
import sys, glob, json, math, random, collections
import numpy as np
from scipy.stats import hypergeom, wilcoxon
sys.path.insert(0, ".")
from kgsim import KG

GMT = sys.argv[1]
CFGS = sys.argv[2].split(",") if len(sys.argv) > 2 else ["kg|rf", "kg|xgb", "seek|rf", "seek|xgb", "hyb|rf", "hyb|xgb"]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"])
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
Xpy = K["Xpy"][:, keep]

# filtered annotations (no IEA, not assigned by Reactome) and symbol -> accession
filt = collections.defaultdict(set); sym2acc = collections.defaultdict(set)
for l in open("godata/goa_human.gaf"):
    if l.startswith("!"): continue
    f = l.rstrip("\n").split("\t")
    if len(f) < 15: continue
    sym2acc[f[2]].add(f[1])
    if "NOT" in f[3] or f[6] == "IEA" or f[14] == "Reactome": continue
    filt[f[1]].add("GO_" + f[4].split(":")[1])
U = sorted(filt); NU = len(U)
asp_has = np.array([[bool(filt[p] & sub[a]) for a in range(len(A))] for p in U])  # NU x 51
uidx = {p: i for i, p in enumerate(U)}
bg = asp_has.sum(0)

sets = []
for l in open(f"{GMT}_msigdb751.gmt"):
    f = l.rstrip("\n").split("\t")
    mem = {a for s in f[2:] for a in (sym2acc.get(s) or {s})} & set(uidx)   # members may be gene symbols or UniProt accessions
    if 10 <= len(mem) <= 300: sets.append((f[0], mem))
member_of = collections.defaultdict(list)
for j, (_, mem) in enumerate(sets):
    for p in mem: member_of[p].append(j)
print(GMT, "gene sets kept", len(sets), "universe", NU, flush=True)

relcache = {}
def relevance(j, p1, p2):
    key = (j, p1, p2)
    if key in relcache: return relcache[key]
    M = [uidx[p] for p in sets[j][1] if p not in (p1, p2)]
    cnt = asp_has[M].sum(0); n = len(M)
    pv = np.array([hypergeom.sf(cnt[a] - 1, NU, bg[a], n) if bg[a] > 0 else 1.0 for a in range(len(A))])
    order = np.argsort(pv); q = np.empty_like(pv); m = len(pv); prev = 1.0
    for rank, a in reversed(list(enumerate(order))):
        prev = min(prev, pv[a] * m / (rank + 1)); q[a] = prev
    relcache[key] = (q < 0.05, np.minimum(-np.log10(np.maximum(pv, 1e-300)), 10.0))
    return relcache[key]

def players(p1, p2):
    t1, t2 = kg.ann[p1], kg.ann[p2]
    return [i for i in range(len(A)) if t1 & sub[i] and t2 & sub[i]]
def ndcg3(order, gain):
    dcg = sum(gain[a] / math.log2(r + 2) for r, a in enumerate(order[:3]))
    ideal = sorted((gain[a] for a in order), reverse=True)
    idcg = sum(g / math.log2(r + 2) for r, g in enumerate(ideal[:3]))
    return dcg / idcg if idcg > 0 else 0.0

out = {}; cases = []
for cfg in CFGS:
    pipe, model = cfg.split("|"); recs = []
    for fn in sorted(glob.glob(f"rec_{pipe}_{model}_f*.json")): recs += json.load(open(fn))["records"]
    methods = [k[4:] for k in recs[0] if k.startswith("phi_")]
    res = collections.defaultdict(lambda: collections.defaultdict(list)); n_tp = n_shared = 0
    for r in recs:
        if not (r["y"] == 1 and r["pred"] == 1): continue
        n_tp += 1
        p1, p2 = keys[r["i"]]
        shared = set(member_of.get(p1, ())) & set(member_of.get(p2, ()))
        if not shared: continue
        n_shared += 1
        j = min(shared, key=lambda j: len(sets[j][1]))
        rel, gain = relevance(j, p1, p2)
        # players: KGsim2vec players for kg; for seek/hyb, aspects with non-zero KAS or LOO attribution (= SEEK players)
        pl = players(p1, p2) if pipe.startswith("kg") else sorted({A.index(a) for m in ("D", "LOO") for a in r[f"phi_{m}"]})
        nrel = int(sum(rel[i] for i in pl))
        if len(pl) < 2 or nrel == 0 or nrel == len(pl): continue
        rr = random.Random(r["i"])
        orders = {}
        for m in methods:
            phi = r[f"phi_{m}"]; tb = {i: rr.random() for i in pl}
            orders[m] = sorted(pl, key=lambda i: (-phi.get(A[i], 0.0), tb[i]))
        if pipe.startswith("kg"):
            tb = {i: rr.random() for i in pl}
            orders["Sim"] = sorted(pl, key=lambda i: (-Xpy[r["i"], i], tb[i]))
        for m, o in orders.items():
            res[m]["hit1"].append(float(rel[o[0]])); res[m]["p3"].append(float(np.mean([rel[a] for a in o[:3]])))
            res[m]["ndcg3"].append(ndcg3(o, gain))
        res["random"]["hit1"].append(nrel / len(pl))
        perms = [rr.sample(pl, len(pl)) for _ in range(200)]
        res["random"]["p3"].append(float(np.mean([np.mean([rel[a] for a in o[:3]]) for o in perms])))
        res["random"]["ndcg3"].append(float(np.mean([ndcg3(o, gain) for o in perms])))
        if cfg == "kg|rf":
            cases.append(dict(pair=r["pair"], pathway=sets[j][0], size=len(sets[j][1]), players=[kg.label.get(A[i], A[i]) for i in pl],
                              relevant=[kg.label.get(A[i], A[i]) for i in pl if rel[i]],
                              top3={m: [kg.label.get(A[i], A[i]) for i in o[:3]] for m, o in orders.items()}))
    summ = dict(n_tp=n_tp, n_shared=n_shared, n_informative=len(res["random"]["hit1"]), metrics={}, wilcoxon_vs_D={})
    for m, d in res.items():
        summ["metrics"][m] = {k: dict(mean=float(np.mean(v)),
                                      ci95=[float(x) for x in np.percentile([np.mean(np.random.default_rng(b).choice(v, len(v))) for b in range(1000)], [2.5, 97.5])])
                              for k, v in d.items()}
    for m in res:
        if m == "D": continue
        for k in ("hit1", "ndcg3"):
            a, b = np.array(res["D"][k]), np.array(res[m][k])
            summ["wilcoxon_vs_D"][f"{m}|{k}"] = float(wilcoxon(a, b).pvalue) if np.any(a != b) else 1.0
    out[cfg] = summ
    print(cfg, "TP", n_tp, "shared", n_shared, "informative", summ["n_informative"])
    for m in summ["metrics"]:
        mm = summ["metrics"][m]
        print("   %-16s hit@1 %.3f  p@3 %.3f  ndcg@3 %.3f" % (m, mm["hit1"]["mean"], mm["p3"]["mean"], mm["ndcg3"]["mean"]),
              "" if m == "D" else "p(hit1)=%.2g p(ndcg)=%.2g" % (summ["wilcoxon_vs_D"].get(f"{m}|hit1", 1), summ["wilcoxon_vs_D"].get(f"{m}|ndcg3", 1)))
json.dump(dict(gmt=GMT, results=out, cases=cases), open(f"plaus_{GMT}{TAG}.json", "w"))


In [ ]:
%%writefile lore_kg.py
"""F3: LORE (Guidotti et al. 2018), the second post-hoc explainer used by the KGsim2vec paper, on the KGsim2vec view.
Uses the LORE authors' library lore_sa (TabularGeneticGeneratorLore: genetic neighbourhood of 1,000 instances + decision-tree
surrogate). Same pairs as the main runs (first N records of rec_kg_{MODEL}_f{FOLD}_*.json).
LORE gives a factual rule + DT feature importances (unsigned). Aspect score phi_LORE = DT importance, signed toward the
predicted class (so ranking() orders by importance). Fidelity is measured with KAS's operator here ('ks');
the neutral operators (zero, imputation) are applied to all methods in road_impute.py.
usage: python3 lore_kg.py MODEL FOLD N   -> f09/lore_kg_{MODEL}_f{FOLD}.json"""
import sys, json, glob, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
MODEL, FOLD, N = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
sys.argv = ["x", "kg", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
import pandas as pd
from lore_sa.dataset import TabularDataset
from lore_sa.bbox import sklearnBBox
from lore_sa.lore import TabularGeneticGeneratorLore
sel = rng.choice((~m).sum(), 2000, replace=False)
df = pd.DataFrame(X[~m][sel], columns=A); df["y"] = np.where(y[~m][sel] == 1, "1", "0")
class BB(sklearnBBox):
    def __init__(s, c): s.bbox = c
    def predict(s, Z): return np.where(clf.predict(np.asarray(Z, float)) == 1, "1", "0")
    def predict_proba(s, Z): return clf.predict_proba(np.asarray(Z, float))
lo = TabularGeneticGeneratorLore(BB(clf), TabularDataset(df, class_name="y"))
recs = []
for fn in sorted(glob.glob(f"rec_{'kg'}_{MODEL}_f{FOLD}_*.json")): recs += json.load(open(fn))["records"]
h = len(recs) // 2; recs = recs[:N // 2] + recs[h:h + N // 2]; out = []; t0 = time.time()   # records: positives first, then negatives
for n, r in enumerate(recs):
    p1, p2 = keys[r["i"]]; st = (set(kg.ann[p1]), set(kg.ann[p2]), set(), set()); x = feats(st)
    pred = r["pred"]; pl = players_of(st)
    np.random.seed(r["i"]); ts = time.time()
    try: e = lo.explain(x, num_instances=1000)
    except Exception as ex: print("ERR", r["i"], ex, flush=True); continue
    tl = time.time() - ts
    imp = dict(e["feature_importances"] or [])
    phi = np.array([float(imp.get(a, 0.0)) for a in A]) * (1 if pred == 1 else -1)
    prem = [(str(p["attr"]), p["op"], float(p["val"])) for p in e["rule"]["premises"]]
    order = ranking(phi, pred)
    o = dict(i=r["i"], y=r["y"], pred=pred, k=len(pl), time_LORE=tl, fid_LORE=float(e["fidelity"]),
             rule=prem, rule_len=len(prem), rule_aspects=sorted({p[0] for p in prem}),
             rule_players=len({p[0] for p in prem} & {A[j] for j in pl}), n_cf=len(e["counterfactuals"]),
             phi_LORE={A[j]: round(float(phi[j]), 5) for j in np.nonzero(phi)[0]}, cov_LORE=int(len(order) > 0),
             size_LORE=eff_size(phi))
    o["del_LORE"] = deletion_ks(st, order, pred) if order else None
    o["ins_LORE"] = insertion_ks(st, order, pred, pl) if order else None
    out.append(o)
    if n % 10 == 0: print(MODEL, FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, aspects=A, records=out), open(f"f09/lore_kg_{MODEL}_f{FOLD}.json", "w"))
print("done", MODEL, FOLD, round(time.time() - t0))


In [ ]:
%%writefile analyze_lore.py
"""F3 summary: LORE vs KAS and the other explainers on the same pairs (KGsim2vec view), KAS's operator ('ks').
Also writes rec_kglore_{MODEL}_f{F}.json (phi of every method on the LORE pairs) for plaus2.py.
usage: python3 analyze_lore.py MODEL"""
import sys, glob, json
import numpy as np
from scipy.stats import wilcoxon
MODEL = sys.argv[1]
L = {}; folds = {}
for f in sorted(glob.glob(f"f09/lore_kg_{MODEL}_f*.json")):
    d = json.load(open(f))
    for r in d["records"]: L[r["i"]] = r; folds[r["i"]] = d["fold"]
R = {r["i"]: r for f in glob.glob(f"rec_kg_{MODEL}_f*_*.json") for r in json.load(open(f))["records"]}
ids = [i for i in L if i in R]
meths = ["D", "LOO", "FS0", "TreeSHAP", "LIME"]
print(f"== LORE (lore_sa, genetic neighbourhood 1,000) vs others, KGsim2vec view, {MODEL.upper()}, {len(ids)} pairs")
print(f"  LORE surrogate fidelity (local DT vs model on neighbourhood): mean {np.mean([L[i]['fid_LORE'] for i in ids]):.3f}")
print(f"  rule length mean {np.mean([L[i]['rule_len'] for i in ids]):.2f}; empty rule {np.mean([L[i]['rule_len']==0 for i in ids]):.3f}; "
      f"rule mentions >=1 player aspect {np.mean([L[i]['rule_players']>0 for i in ids]):.3f}; "
      f"share of rule aspects that are non-players {np.mean([1-L[i]['rule_players']/max(1,len(L[i]['rule_aspects'])) for i in ids if L[i]['rule_aspects']]):.3f}")
print(f"  coverage (>=1 ranked aspect) {np.mean([L[i]['cov_LORE'] for i in ids]):.3f}; time/pair {np.mean([L[i]['time_LORE'] for i in ids]):.1f} s "
      f"(KAS {np.mean([R[i]['time_D'] for i in ids]):.2f} s)")
def auc(c): return float(np.mean(c)) if c else np.nan
rows = {m: ([auc(R[i][f"del_{m}"]) for i in ids], [auc(R[i][f"ins_{m}"]) for i in ids]) for m in meths}
rows["LORE"] = ([auc(L[i]["del_LORE"]) for i in ids], [auc(L[i]["ins_LORE"]) for i in ids])
rows["random"] = ([auc(R[i]["del_random"]) for i in ids], [auc(R[i]["ins_random"]) for i in ids])
print("  method    deletion AUC (down)  insertion AUC (up)   [operator ks, NaN pairs = no ranked aspect, excluded]")
summ = {}
for m, (de, ins) in rows.items():
    summ[m] = (float(np.nanmean(de)), float(np.nanmean(ins))); print(f"  {m:9s} {np.nanmean(de):.3f}                {np.nanmean(ins):.3f}")
pv = {}
for m in rows:
    if m == "D": continue
    a, b = np.array(rows["D"][0]), np.array(rows[m][0]); ok = ~np.isnan(a) & ~np.isnan(b)
    a2, b2 = np.array(rows["D"][1]), np.array(rows[m][1]); ok2 = ~np.isnan(a2) & ~np.isnan(b2)
    pv[m] = (float(wilcoxon(a[ok], b[ok]).pvalue), float(np.mean(a[ok] < b[ok])), float(wilcoxon(a2[ok2], b2[ok2]).pvalue), float(np.mean(a2[ok2] > b2[ok2])))
    print(f"  KAS vs {m:9s}: deletion p={pv[m][0]:.2g} KAS lower in {pv[m][1]:.2f} of pairs; insertion p={pv[m][2]:.2g} KAS higher in {pv[m][3]:.2f}")
t1 = np.mean([int(bool(R[i]['top3_D']) and bool(L[i]['phi_LORE']) and R[i]['top3_D'][0] == max(L[i]['phi_LORE'], key=lambda a: abs(L[i]['phi_LORE'][a]))) for i in ids])
print(f"  top-1 aspect same as KAS: {t1:.3f}")
json.dump(dict(n=len(ids), summ=summ, p=pv, top1_same_D=float(t1)), open(f"f09/lore_{MODEL}_summary.json", "w"))
# merged records for plausibility
by = {}
for i in ids:
    o = {k: R[i][k] for k in ("i", "pair", "y", "pred")}; o.update({f"phi_{m}": R[i][f"phi_{m}"] for m in meths}); o["phi_LORE"] = L[i]["phi_LORE"]
    by.setdefault(folds[i], []).append(o)
A = json.load(open(glob.glob(f"rec_kg_{MODEL}_f1_*.json")[0]))["aspects"]
for f, recs in by.items(): json.dump(dict(aspects=A, records=recs), open(f"rec_kglore_{MODEL}_f{f}.json", "w"))


In [ ]:
%%writefile road_impute.py
"""F1: ROAD-style evaluation (Rong et al. 2022 idea: remove features by IMPUTATION, keep the trained model, no retraining)
on the KGsim2vec view. Removing aspects S of a pair = replace x_S by a draw from the Gaussian conditional
p(x_S | x_rest) fitted on the fold's training features (noisy linear imputation, clipped at 0), and ask the ORIGINAL
fold model. No retraining -> no class leakage through the mask; imputation keeps the input close to the data manifold
(checked below with a discriminator). This operator is neutral: it is neither KAS's (delete annotations + recompute)
nor FS0's (set to 0). On the 2,200-pair sets a third operator, knn donor imputation (copy the removed values from the
training pair nearest in the kept features), keeps every value a real observed one. Both directions: MoRF (remove most relevant first, lower curve = better) and LeRF (remove least
relevant first, higher curve = better), k = 1..5 aspects, plus the zero operator for reference.
usage: python3 road_impute.py MODEL SRC   SRC=all (47,142 pairs, all_kg_MODEL_f*.json) | rec (2,200 main-run pairs,
       plus LORE scores from f09/lore_kg_MODEL_f*.json for the pairs LORE explained)  -> f09/road_{MODEL}_{SRC}.npz/.json"""
import sys, glob, json, random, time
import numpy as np
sys.path.insert(0, "."); sys.path.insert(0, "ExplainablePPI")
from kgsim import KG
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
MODEL, SRC = sys.argv[1], sys.argv[2]; KMAX = 5; NJ = int(__import__("os").environ.get("NJOBS", "4"))
K_ = np.load("feats_kg.npz"); keys = [tuple(k) for k in K_["keys"]]; aspects = list(K_["aspects"]); y = K_["y"]
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
X0 = K_["Xpy"][:, keep]; nA = len(A)
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
P = "ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds = [[int(l) for l in open(P + ("1(complete)" if f == 1 else str(f)) + ".txt") if l.strip()] for f in range(1, 11)]
rec = {}
if SRC == "all":
    for f in sorted(glob.glob(f"all_kg_{MODEL}_f*.json")):
        for r in json.load(open(f))["records"]: rec[r["i"]] = r
else:
    for f in sorted(glob.glob(f"rec_kg_{MODEL}_f*_*.json")):
        for r in json.load(open(f))["records"]: rec[r["i"]] = {k: v for k, v in r.items() if k in ("i", "y", "pred") or k.startswith("phi_")}
    for f in sorted(glob.glob(f"f09/lore_kg_{MODEL}_f*.json")):
        for r in json.load(open(f))["records"]:
            if r["i"] in rec: rec[r["i"]]["phi_LORE"] = r["phi_LORE"]
methods = sorted({k[4:] for r in rec.values() for k in r if k.startswith("phi_")})
print(SRC, MODEL, len(rec), "pairs, methods", methods, flush=True)
orders = [(m, d) for m in methods for d in ("MoRF", "LeRF")] + [("random", "MoRF")]
OPS = ("imp", "zero", "knn") if SRC == "rec" else ("imp", "zero")   # knn donor imputation only on the 2,200-pair sets (cost)
ids = sorted(rec); pos = {i: n for n, i in enumerate(ids)}
PR = {(m, d, op): np.full((len(ids), KMAX), np.nan, np.float32) for m, d in orders for op in OPS}
PR0 = np.zeros(len(ids), np.float32); FOLD = np.zeros(len(ids), np.int8); PRED = np.zeros(len(ids), np.int8)
HAS = {m: np.zeros(len(ids), bool) for m in methods}; HAS["random"] = np.ones(len(ids), bool)
disc = {"imp": [], "zero": [], "knn": [], "orig": []}
t0 = time.time()
for fi, te in enumerate(folds):
    m_ = np.zeros(len(y), bool); m_[te] = True
    clf = (RandomForestClassifier(100, n_jobs=NJ, random_state=0) if MODEL == "rf"
           else XGBClassifier(n_estimators=300, n_jobs=NJ, random_state=0, eval_metric="logloss")).fit(X0[~m_], y[~m_])
    DON = X0[~m_][np.random.default_rng(fi).choice((~m_).sum(), 5000, replace=False)]   # donor pool for knn imputation
    mu = X0[~m_].mean(0); Sig = np.cov(X0[~m_], rowvar=False) + 1e-6 * np.eye(nA); Lam = np.linalg.inv(Sig)
    rows, slots = [], []
    for i in te:
        if i not in rec: continue
        r = rec[i]; n = pos[i]; FOLD[n] = fi + 1; x = X0[i]; t1, t2 = kg.ann[keys[i][0]], kg.ann[keys[i][1]]
        pl = [a for a in range(nA) if t1 & sub[a] and t2 & sub[a]]; pred = r["pred"]; PRED[n] = pred
        sgn = 1 if pred == 1 else -1; rr = random.Random(i); tb = {a: rr.random() for a in pl}
        rows.append(x); slots.append((n, None, None, None))
        cache = {}
        for m, d in orders:
            if m == "random": order = random.Random(i + 7).sample(pl, len(pl))
            else:
                if f"phi_{m}" not in r: continue
                HAS[m][n] = True; phi = r[f"phi_{m}"]; s = {a: sgn * phi.get(A[a], 0.0) for a in pl}
                order = sorted(pl, key=lambda a: (-s[a], tb[a])) if d == "MoRF" else sorted(pl, key=lambda a: (s[a], tb[a]))
            for k in range(1, KMAX + 1):
                S = tuple(sorted(order[:k]))
                if not S: z = {op: x for op in OPS}
                elif S in cache: z = cache[S]
                else:
                    Sl = list(S); Kl = [j for j in range(nA) if j not in S]
                    C = np.linalg.inv(Lam[np.ix_(Sl, Sl)]); mean = mu[Sl] - C @ Lam[np.ix_(Sl, Kl)] @ (x[Kl] - mu[Kl])
                    g = np.random.default_rng(i * 131 + hash(S) % 100003)
                    zi = x.copy(); zi[Sl] = np.clip(g.multivariate_normal(mean, C, method="cholesky"), 0, None)
                    zz = x.copy(); zz[Sl] = 0; z = cache[S] = {"imp": zi, "zero": zz}
                    if "knn" in OPS:   # copy x_S from the training pair nearest in the kept features (real, on-manifold values)
                        nn = int(np.argmin(((DON[:, Kl] - x[Kl]) ** 2).sum(1))); zk = x.copy(); zk[Sl] = DON[nn, Sl]; z["knn"] = zk
                    if m == "random" and k == 2 and len(disc["imp"]) < 30000:
                        for op in OPS: disc[op].append(z[op])
                        disc["orig"].append(x)
                for op in OPS: rows.append(z[op]); slots.append((n, (m, d, op), k - 1, None))
    p = clf.predict_proba(np.array(rows))[:, 1]
    for (n, key, kk, _), q in zip(slots, p):
        q = q if PRED[n] == 1 else 1 - q
        if key is None: PR0[n] = q
        else: PR[key][n, kk] = q
    print("fold", fi + 1, len(rows), "rows", round(time.time() - t0), "s", flush=True)
# discriminator: can a classifier tell perturbed inputs from real ones? (0.5 = indistinguishable)
dres = {}
for op in OPS:
    Z = np.vstack([np.array(disc["orig"]), np.array(disc[op])]); lab = np.r_[np.zeros(len(disc["orig"])), np.ones(len(disc[op]))]
    idx = np.random.default_rng(0).permutation(len(lab)); h = len(idx) // 2
    dc = RandomForestClassifier(100, n_jobs=NJ, random_state=0, min_samples_leaf=5).fit(Z[idx[:h]], lab[idx[:h]])
    dres[op] = float(roc_auc_score(lab[idx[h:]], dc.predict_proba(Z[idx[h:]])[:, 1]))
print("discriminator AUC", dres, flush=True)
np.savez_compressed(f"f09/road_{MODEL}_{SRC}.npz", ids=np.array(ids), fold=FOLD, pred=PRED, p0=PR0,
                    **{f"{m}|{d}|{op}": v for (m, d, op), v in PR.items()}, **{f"has|{m}": v for m, v in HAS.items()})
json.dump(dict(model=MODEL, src=SRC, n=len(ids), methods=methods, disc_auc=dres), open(f"f09/road_{MODEL}_{SRC}.json", "w"))
print("FINISHED", round(time.time() - t0))


In [ ]:
%%writefile analyze_road.py
"""Summarise road_impute.py output. usage: python3 analyze_road.py MODEL SRC [SUBSET_METHOD]
AUC = mean predicted-class probability over k=1..5 removed aspects (MoRF: lower = better; LeRF: higher = better);
gap = LeRF - MoRF (higher = better); flip@1 = share of pairs whose class flips after removing 1 aspect (MoRF).
Pairs restricted to those explained by every method shown (SUBSET_METHOD, e.g. LORE, restricts further).
Wilcoxon signed-rank, two-sided (paired over pairs), KAS(D) vs each method; direction = folds where KAS is better."""
import sys, json
import numpy as np
from scipy.stats import wilcoxon
MODEL, SRC = sys.argv[1], sys.argv[2]; SUBM = sys.argv[3] if len(sys.argv) > 3 else None
Z = np.load(f"f09/road_{MODEL}_{SRC}.npz"); meta = json.load(open(f"f09/road_{MODEL}_{SRC}.json"))
methods = meta["methods"]
mask = np.ones(len(Z["ids"]), bool)
for m in methods:
    if SUBM is None and m == "LORE": continue
    mask &= Z[f"has|{m}"]
show = [m for m in methods if SUBM is not None or m != "LORE"]
fold = Z["fold"][mask]
print(f"== ROAD-style imputation, KGsim2vec view, {MODEL.upper()}, {mask.sum()} pairs; discriminator AUC " + " ".join(f"{k} {v:.3f}" for k, v in meta['disc_auc'].items()) + "")
out = {}
for op in [o for o in ("imp", "knn", "zero") if f"D|MoRF|{o}" in Z.files]:
    print(f"-- operator {op}")
    print("  method    | MoRF AUC (down) | LeRF AUC (up) | gap (up) | MoRF p@k1 k2 k3 | flip@1")
    stats = {}
    for m in show + ["random"]:
        mo = Z[f"{m}|MoRF|{op}"][mask]; mo = np.where(np.isnan(mo), np.nan, mo)
        mor = np.nanmean(mo, 1)
        if m != "random":
            le = Z[f"{m}|LeRF|{op}"][mask]; ler = np.nanmean(le, 1); gap = ler - mor
        else: ler = gap = np.full_like(mor, np.nan)
        stats[m] = (mor, ler, gap)
        fl = np.mean(mo[:, 0] < 0.5)
        print(f"  {m:9s} | {np.nanmean(mor):.3f}           | {np.nanmean(ler):.3f}         | {np.nanmean(gap):.3f}    | "
              f"{np.nanmean(mo[:,0]):.3f} {np.nanmean(mo[:,1]):.3f} {np.nanmean(mo[:,2]):.3f} | {fl:.3f}")
    for m in show:
        if m == "D": continue
        res = {}
        for j, (nm, alt) in enumerate((("MoRF", "less"), ("LeRF", "greater"), ("gap", "greater"))):
            a, b = stats["D"][j], stats[m][j]; ok = ~np.isnan(a) & ~np.isnan(b)
            p = wilcoxon(a[ok], b[ok]).pvalue if np.any(a[ok] != b[ok]) else 1.0   # two-sided; direction from the fold count
            fw = sum((np.nanmean(a[fold == f]) < np.nanmean(b[fold == f])) if nm == "MoRF" else (np.nanmean(a[fold == f]) > np.nanmean(b[fold == f]))
                     for f in range(1, 11) if np.any(fold == f))
            res[nm] = (float(p), int(fw))
        out[f"{op}|{m}"] = res
        print(f"  KAS vs {m:9s}: " + "  ".join(f"{k} p={v[0]:.2g} folds KAS better {v[1]}/10" for k, v in res.items()))
    out[op] = {m: [float(np.nanmean(s)) for s in stats[m]] for m in stats}
json.dump(dict(n=int(mask.sum()), disc=meta["disc_auc"], res=out), open(f"f09/road_{MODEL}_{SRC}{'_'+SUBM if SUBM else ''}_summary.json", "w"))


In [ ]:
%%writefile sanity.py
"""F2: sanity checks for the explainers (Adebayo et al. 2018, 'Sanity checks for saliency maps') + seed stability.
- model randomisation (data randomisation test): RF trained on PERMUTED labels; a faithful explainer must change a lot
  (low rank correlation with the explanation of the real model).
- seed stability: RF retrained with random_state 1..4 (same data); explanations of an equally accurate model should
  stay similar (high rank correlation). This replaces the 'Consistency = 1 by construction' cells of the VXAI table.
Same 300 pairs (150 pos + 150 neg) of fold 1 as the main run; reference explanation = the recorded one (RF seed 0).
usage: python3 sanity.py VIEW   VIEW in {kg, seek}   -> f09/sanity_{VIEW}.json"""
import sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
VIEW = sys.argv[1]
sys.argv = ["x", VIEW, "rf", "1", "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
from seeknat import native
from scipy.stats import spearmanr
from sklearn.metrics import f1_score
R = json.load(open(f"rec_{VIEW}_rf_f1_400.json"))["records"]; R = R[:150] + R[200:350]
if VIEW == "seek":
    SX = {r["i"]: r for r in json.load(open("rec_seekx_rf_f1.json"))["records"]}
names = list(METHODS) + (["SEEKN", "SEEKS"] if VIEW == "seek" else [])
def phis(st):
    out = {nm: f(st, 0) for nm, f in METHODS.items()}
    if VIEW == "seek":
        pN, pS, *_ = native(st, sk, sub_sk, proba, len(A)); out["SEEKN"], out["SEEKS"] = pN, pS
    return out
def ref_phi(r, nm):
    src = SX[r["i"]] if nm in ("SEEKN", "SEEKS") else r
    return np.array([src[f"phi_{nm}"].get(a, 0.0) for a in A])
Ytr = y[~m].copy()
models = {f"seed{s}": (s, Ytr) for s in (1, 2, 3, 4)}
models["randlabel"] = (0, np.random.default_rng(0).permutation(Ytr))
res = {}; t0 = time.time()
for mname, (seed, ytr) in models.items():
    clf = RandomForestClassifier(100, n_jobs=1, random_state=seed).fit(X[~m], ytr)
    tree = shap.TreeExplainer(clf, data=bg, feature_perturbation="interventional")
    waf = float(f1_score(y[m], clf.predict(X[m]), average="weighted"))
    rows = []
    for r in R:
        p1, p2 = keys[r["i"]]
        st = (set(kg.ann[p1]), set(kg.ann[p2]), set(sk.ann[p1]) - CX, set(sk.ann[p2]) - CX)
        pl = players_of(st); pred = int(proba(feats(st))[0] > 0.5)
        ph = phis(st); row = dict(i=r["i"], k=len(pl), pred_same=int(pred == r["pred"]))
        for nm in names:
            a, b = ref_phi(r, nm), ph[nm]
            oa, ob = ranking(a, r["pred"]), ranking(b, pred)
            row[f"top1_{nm}"] = int(bool(oa) and bool(ob) and oa[0] == ob[0])
            row[f"jac3_{nm}"] = jacc(oa, ob)
            if len(pl) >= 3:
                va, vb = a[pl], b[pl]
                row[f"rho_{nm}"] = float(spearmanr(va, vb).correlation) if np.std(va) > 0 and np.std(vb) > 0 else None
        rows.append(row)
    res[mname] = dict(waf=waf, rows=rows)
    print(VIEW, mname, "WAF %.3f" % waf, round(time.time() - t0), "s", flush=True)
    json.dump(dict(view=VIEW, methods=names, base_waf=None, res=res), open(f"f09/sanity_{VIEW}.json", "w"))
print("FINISHED")


In [ ]:
%%writefile analyze_sanity.py
"""Summarise sanity.py (F2) + RF-vs-XGB agreement on the same pairs.
usage: python3 analyze_sanity.py VIEW
Per method: mean Spearman rho (players, k>=3), top-1 agreement and top-3 Jaccard between the reference explanation
(RF seed 0) and (a) RF with another seed [want HIGH], (b) RF trained on permuted labels [want LOW]."""
import sys, json
import numpy as np
VIEW = sys.argv[1]
d = json.load(open(f"f09/sanity_{VIEW}.json")); ms = d["methods"]; out = {}
def agg(rows, key):
    v = [r[key] for r in rows if r.get(key) is not None and not np.isnan(r[key])]; return (float(np.mean(v)) if v else float("nan")), len(v)
print(f"== sanity checks, {VIEW} view, RF, fold 1, 300 pairs")
for mn, r in d["res"].items(): print(f"  model {mn}: WAF {r['waf']:.3f}, same predicted class as reference {np.mean([x['pred_same'] for x in r['rows']]):.3f}")
print("  method   | seeds 1-4: rho / top1 / jac3   | permuted labels: rho / top1 / jac3 | drop in rho")
for m in ms:
    seeds = [x for s in ("seed1", "seed2", "seed3", "seed4") if s in d["res"] for x in d["res"][s]["rows"]]
    rnd = d["res"].get("randlabel", {"rows": []})["rows"]
    a = [agg(seeds, f"{k}_{m}")[0] for k in ("rho", "top1", "jac3")]; b = [agg(rnd, f"{k}_{m}")[0] for k in ("rho", "top1", "jac3")]
    out[m] = dict(seed=a, rand=b)
    print(f"  {m:8s} | {a[0]:.3f} / {a[1]:.3f} / {a[2]:.3f}          | {b[0]:.3f} / {b[1]:.3f} / {b[2]:.3f}              | {a[0]-b[0]:.3f}")
# RF vs XGB on the same 2,200 pairs (implementation invariance across model families; same data, similar accuracy)
import glob
from scipy.stats import spearmanr
src = "kg" if VIEW == "kg" else "seek"
rf = {r["i"]: r for f in glob.glob(f"rec_{src}_rf_f*_*.json") for r in json.load(open(f))["records"]}
xg = {r["i"]: r for f in glob.glob(f"rec_{src}_xgb_f*_*.json") for r in json.load(open(f))["records"]}
common = sorted(set(rf) & set(xg)); A = json.load(open(glob.glob(f"rec_{src}_rf_f1_*.json")[0]))["aspects"]
print(f"  RF vs XGB, same {len(common)} pairs (both models predict the same class in "
      f"{np.mean([rf[i]['pred'] == xg[i]['pred'] for i in common]):.3f}):")
out["rf_vs_xgb"] = {}
for m in [k[4:] for k in rf[common[0]] if k.startswith("phi_")]:
    rh, t1 = [], []
    for i in common:
        a, b = rf[i][f"phi_{m}"], xg[i].get(f"phi_{m}")
        if b is None: continue
        keys_ = sorted(set(a) | set(b))
        if len(keys_) >= 3:
            va = [a.get(k, 0) for k in keys_]; vb = [b.get(k, 0) for k in keys_]
            if np.std(va) > 0 and np.std(vb) > 0:
                c = spearmanr(va, vb).correlation
                if not np.isnan(c): rh.append(c)
        if rf[i]["pred"] == xg[i]["pred"]:
            ta, tb = rf[i].get(f"top3_{m}") or [], xg[i].get(f"top3_{m}") or []
            t1.append(int(bool(ta) and bool(tb) and ta[0] == tb[0]))
    out["rf_vs_xgb"][m] = (float(np.mean(rh)), float(np.mean(t1)))
    print(f"    {m:8s} rho {np.mean(rh):.3f}  top-1 same {np.mean(t1):.3f}")
json.dump(out, open(f"f09/sanity_{VIEW}_summary.json", "w"))


In [ ]:
%%writefile kas_fine.py
"""F4: KAS at a finer granularity, with the trained model unchanged (KAS = one method parameterised by the partition).
Players = the direct-child subtrees of each of the 51 aspects (GO depth 2) + one residual unit per aspect holding the
aspect term itself. A unit is a player of a pair when it lies in an aspect-level player and holds >=1 direct annotation of either
protein (so removing every fine player removes exactly what removing every aspect player removes, and the fine
attributions sum to the same total as aspect-level KAS). Value function as KAS: delete both proteins' annotations in the removed units, recompute the 51 KGsim2vec
features, read the model. Exact Shapley for k<=8, else 100 permutations. Compared with LOO and random at the same
granularity (deletion/insertion, operator ks), and checked for consistency with aspect-level KAS (sum of fine phi
inside an aspect vs the aspect's phi).
usage: python3 kas_fine.py MODEL FOLD N   -> f09/fine_kg_{MODEL}_f{FOLD}.json"""
import sys, json, time, random, itertools, math
import numpy as np
MODEL, FOLD, N = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
sys.argv = ["x", "kg", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
from scipy.stats import spearmanr
units, unit_asp, unit_name = [], [], []
for ai, a in enumerate(A):
    for c in sorted(kg.children.get(a, ())):
        if c in sub_kg[ai]: units.append(kg.desc(c) | {c}); unit_asp.append(ai); unit_name.append(c)
    units.append({a}); unit_asp.append(ai); unit_name.append(a + "(self)")
U = len(units); print("fine units", U, flush=True)
def players_f(st):   # units inside an aspect-level player that hold >=1 direct annotation of either protein
    k1, k2 = st[0], st[1]; pa = set(players_of(st)); return [u for u in range(U) if unit_asp[u] in pa and ((k1 | k2) & units[u])]
def remove_f(st, S):
    if not S: return st
    r = set().union(*[units[u] for u in S]); return (st[0] - r, st[1] - r, st[2], st[3])
def shapley_f(st, pl, seed):
    k = len(pl); phi = np.zeros(U)
    if k == 0: return phi
    vb = lambda subs: proba(np.array([feats(remove_f(st, S)) for S in subs]))
    if k <= EXACT_MAX:
        subsets = [frozenset(c) for r in range(k + 1) for c in itertools.combinations(pl, r)]; vals = dict(zip(subsets, vb(subsets)))
        for i in pl:
            others = [p for p in pl if p != i]
            for r in range(k):
                w = math.factorial(r) * math.factorial(k - r - 1) / math.factorial(k)
                for c in itertools.combinations(others, r): S = frozenset(c); phi[i] += w * (vals[S] - vals[S | {i}])
        return phi
    rr = random.Random(seed); perms = [rr.sample(pl, k) for _ in range(M_PERM)]; subsets = set()
    for p in perms:
        for t in range(k + 1): subsets.add(frozenset(p[:t]))
    subsets = list(subsets); vals = dict(zip(subsets, vb(subsets)))
    for p in perms:
        for t, i in enumerate(p): phi[i] += vals[frozenset(p[:t])] - vals[frozenset(p[:t + 1])]
    return phi / M_PERM
def curves(st, order, pred, pl):
    Zd = [feats(remove_f(st, order[:k])) for k in range(1, TOPK + 1)]
    Zi = [feats(remove_f(st, [u for u in pl if u not in order[:k]])) for k in range(0, TOPK + 1)]
    p = proba(np.array(Zd + Zi)); p = p if pred == 1 else 1 - p
    return list(p[:TOPK]), list(p[TOPK:])
R = json.load(open(f"rec_kg_{MODEL}_f{FOLD}_" + ("400" if FOLD == 1 else "200") + ".json"))["records"]
h = len(R) // 2; R = R[:N // 2] + R[h:h + N // 2]; out = []; t0 = time.time()
for n, r in enumerate(R):
    p1, p2 = keys[r["i"]]; st = (set(kg.ann[p1]), set(kg.ann[p2]), set(), set()); pred = r["pred"]
    pl = players_f(st); ts = time.time(); phi = shapley_f(st, pl, 0); tk = time.time() - ts
    v = proba(np.array([feats(st)] + [feats(remove_f(st, [u])) for u in pl])); loo = np.zeros(U)
    for j, u in enumerate(pl): loo[u] = v[0] - v[j + 1]
    o = dict(i=r["i"], y=r["y"], pred=pred, k_aspect=r["k"], k_fine=len(pl), time_fine=tk, time_aspect=r["time_D"])
    for nm, f in (("Dfine", phi), ("LOOfine", loo)):
        od = ranking(f, pred); o[f"del_{nm}"], o[f"ins_{nm}"] = curves(st, od, pred, pl) if od else (None, None)
        o[f"phi_{nm}"] = {unit_name[u]: round(float(f[u]), 5) for u in np.nonzero(f)[0]}
    ro = [int(x) for x in np.random.default_rng(r["i"]).permutation(pl)]
    o["del_randomfine"], o["ins_randomfine"] = curves(st, ro, pred, pl) if ro else (None, None)
    # consistency with aspect-level KAS: aggregate fine phi to aspects
    agg = np.zeros(len(A))
    for u in pl: agg[unit_asp[u]] += phi[u]
    da = np.array([r["phi_D"].get(a, 0.0) for a in A]); asp_pl = sorted({unit_asp[u] for u in pl})
    o["rho_agg_vs_aspect"] = float(spearmanr(agg[asp_pl], da[asp_pl]).correlation) if len(asp_pl) >= 3 and np.std(agg[asp_pl]) > 0 and np.std(da[asp_pl]) > 0 else None
    oa, of = ranking(da, pred), ranking(phi, pred)
    o["top1_fine_in_top1_aspect"] = int(bool(oa) and bool(of) and unit_asp[of[0]] == oa[0]) if oa and of else None
    o["top1_fine"] = unit_name[of[0]] if of else None; o["top1_fine_label"] = kg.label.get(unit_name[of[0]].split("(")[0], "") if of else None
    o["sum_fine"], o["sum_aspect"] = float(phi.sum()), float(da.sum())
    out.append(o)
    if n % 10 == 0: print(MODEL, FOLD, n, "k_fine", len(pl), round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, n_units=U, records=out), open(f"f09/fine_kg_{MODEL}_f{FOLD}.json", "w"))
print("done", round(time.time() - t0))


In [ ]:
%%writefile analyze_fine.py
"""F4 summary. usage: python3 analyze_fine.py MODEL"""
import sys, json
import numpy as np
from scipy.stats import wilcoxon
MODEL = sys.argv[1]; d = json.load(open(f"f09/fine_kg_{MODEL}_f1.json")); R = d["records"]
print(f"== KAS with finer players (depth-2 GO subtrees + residual unit per aspect; {d['n_units']} units), KGsim2vec view, {MODEL.upper()}, fold 1, {len(R)} pairs")
print(f"  players per pair: aspect level {np.mean([r['k_aspect'] for r in R]):.1f}, fine level {np.mean([r['k_fine'] for r in R]):.1f} (median {np.median([r['k_fine'] for r in R]):.0f}, "
      f"exact Shapley for {np.mean([r['k_fine'] <= 8 for r in R]):.2f} of pairs)")
print(f"  time per pair: fine {np.mean([r['time_fine'] for r in R]):.2f} s, aspect {np.mean([r['time_aspect'] for r in R]):.2f} s")
print(f"  efficiency: max |sum fine phi - sum aspect phi| = {max(abs(r['sum_fine'] - r['sum_aspect']) for r in R):.1e}")
rho = [r["rho_agg_vs_aspect"] for r in R if r["rho_agg_vs_aspect"] is not None]
t1 = [r["top1_fine_in_top1_aspect"] for r in R if r["top1_fine_in_top1_aspect"] is not None]
print(f"  fine phi summed per aspect vs aspect-level phi: Spearman {np.mean(rho):.3f} (n={len(rho)}); top-1 fine unit inside the top-1 aspect {np.mean(t1):.3f}")
auc = lambda c: float(np.mean(c)) if c else np.nan
rows = {m: (np.array([auc(r[f"del_{m}"]) for r in R]), np.array([auc(r[f"ins_{m}"]) for r in R])) for m in ("Dfine", "LOOfine", "randomfine")}
out = {}
print("  method       deletion AUC (down)  insertion AUC (up)   [operator ks, k = 1..5 fine units]")
for m, (de, ins) in rows.items():
    out[m] = (float(np.nanmean(de)), float(np.nanmean(ins))); print(f"  {m:12s} {np.nanmean(de):.3f}                {np.nanmean(ins):.3f}")
for m in ("LOOfine", "randomfine"):
    a, b = rows["Dfine"][0], rows[m][0]; ok = ~np.isnan(a) & ~np.isnan(b)
    a2, b2 = rows["Dfine"][1], rows[m][1]; ok2 = ~np.isnan(a2) & ~np.isnan(b2)
    pd_, pi_ = wilcoxon(a[ok], b[ok]).pvalue, wilcoxon(a2[ok2], b2[ok2]).pvalue
    out[f"p_{m}"] = (float(pd_), float(np.mean(a[ok] < b[ok])), float(pi_), float(np.mean(a2[ok2] > b2[ok2])))
    print(f"  KAS-fine vs {m}: deletion p={pd_:.2g} (KAS lower in {np.mean(a[ok] < b[ok]):.2f}), insertion p={pi_:.2g} (KAS higher in {np.mean(a2[ok2] > b2[ok2]):.2f})")
from collections import Counter
print("  most frequent top-1 fine units:", Counter(r["top1_fine_label"] for r in R if r["top1_fine_label"]).most_common(8))
json.dump(out, open(f"f09/fine_{MODEL}_summary.json", "w"))


## 1. Đặc trưng và hàm chạy song song

In [ ]:
import os
if not os.path.exists('feats_kg.npz'):
    !python build_feats.py && cp feats.npz feats_kg.npz
if not os.path.exists('feats_seek.npz'):
    !python build_seek_feats.py
print('feats_kg.npz', os.path.exists('feats_kg.npz'), '| feats_seek.npz', os.path.exists('feats_seek.npz'))

In [ ]:
import os, subprocess, time, glob
from concurrent.futures import ThreadPoolExecutor
def run_jobs(jobs, workers=None):
    """jobs = [(output_file, [command...], log_file)]; a job whose output already exists is skipped."""
    t0 = time.time()
    def one(j):
        out, cmd, log = j
        if os.path.exists(out): return out + ': đã có, bỏ qua'
        t = time.time(); r = subprocess.run(cmd, capture_output=True, text=True); open(log, 'w').write(r.stdout + r.stderr)
        return out + (f': xong sau {(time.time()-t)/60:.1f} phút' if r.returncode == 0 else ': LỖI\n' + r.stderr[-1500:])
    with ThreadPoolExecutor(max_workers=workers or os.cpu_count()) as ex:
        for i, msg in enumerate(ex.map(one, jobs)): print(f'[{i+1}/{len(jobs)}, {(time.time()-t0)/60:.0f} phút]', msg, flush=True)

## 2. Dữ liệu đầu vào từ các lần chạy trước
Cần: `rec_kg_{rf,xgb}_f*.json` (2,200 cặp mỗi mô hình), `rec_seek_rf_f1_400.json`, `rec_seekx_rf_f1.json` và `all_kg_rf_f*.json` (47,142 cặp). Ô dưới chỉ chạy lại phần còn thiếu (nếu thiếu tất cả thì mất vài giờ).

In [ ]:
run_jobs([(f'rec_{p}_{m}_f{f}_{400 if f == 1 else 200}.json', ['python', 'explain_v2.py', p, m, str(f), str(400 if f == 1 else 200)],
           f'log_rec_{p}_{m}_{f}.txt') for p, m in [('kg', 'rf'), ('kg', 'xgb'), ('seek', 'rf')] for f in range(1, 11) if p == 'kg' or f == 1])
run_jobs([('rec_seekx_rf_f1.json', ['python', 'seek_native.py', 'rf', '1'], 'log_e1_rf_1.txt')])
if len(glob.glob('all_kg_rf_f*.json')) < 10:
    NPARTS = max(1, os.cpu_count() // 2)
    run_jobs([(f'all_kg_rf_f{f}_p{p}.json' if NPARTS > 1 else f'all_kg_rf_f{f}.json', ['python', 'explain_all.py', 'rf', str(f), str(p), str(NPARTS)],
               f'log_all_kg_{f}_{p}.txt') for f in range(1, 11) for p in range(NPARTS)])
    import json
    for f in range(1, 11):
        parts = sorted(glob.glob(f'all_kg_rf_f{f}_p*.json'))
        if parts and not os.path.exists(f'all_kg_rf_f{f}.json'):
            d = json.load(open(parts[0])); d['records'] = [r for p in parts for r in json.load(open(p))['records']]
            json.dump(d, open(f'all_kg_rf_f{f}.json', 'w'))
print(len(glob.glob('rec_kg_*_f*_*.json')), 'rec_kg files;', len(glob.glob('all_kg_rf_f?.json')) + len(glob.glob('all_kg_rf_f10.json')), 'all_kg_rf files')

## F3. LORE trên view KGsim2vec
60 cặp mỗi fold (30 dương, 30 âm, lấy từ các cặp của lần chạy chính) × 10 fold × RF/XGB = 1,200 cặp. Mỗi cặp: lân cận di truyền 1,000 mẫu, cây quyết định thay thế, luật factual và counterfactual. Điểm aspect = độ quan trọng trong cây (không dấu, quy về phía lớp dự đoán).

In [ ]:
run_jobs([(f'f09/lore_kg_{m}_f{f}.json', ['python', 'lore_kg.py', m, str(f), '60'], f'f09/log_lore_{m}_{f}.txt')
          for m in ['rf', 'xgb'] for f in range(1, 11)])
!python analyze_lore.py rf | tee f09/f3_lore_rf.txt
!python analyze_lore.py xgb | tee f09/f3_lore_xgb.txt
if not os.path.exists('REACTOME_msigdb751.gmt'):
    !python get_msigdb.py
!python plaus2.py REACTOME 'kglore|rf,kglore|xgb' _kglore | tee f09/f3_plaus_reactome.txt

## F1. Đánh giá kiểu ROAD (điền khuyết, không huấn luyện lại)
Xoá k aspect = thay giá trị đặc trưng của chúng bằng một mẫu từ phân phối Gaussian có điều kiện (fit trên dữ liệu train của fold), rồi hỏi **mô hình gốc**. Hai chiều MoRF/LeRF, k = 1..5; kèm toán tử đặt 0 để tham chiếu và một bộ phân biệt (discriminator) kiểm tra đầu vào sau điền khuyết có lệch phân phối không.
Chạy trên 47,142 cặp (RF) và 2,200 cặp (RF, XGB; có LORE trên tập con 1,200 cặp).

In [ ]:
for m, s in [('xgb', 'rec'), ('rf', 'rec'), ('rf', 'all')]:
    if not os.path.exists(f'f09/road_{m}_{s}.json'):
        !NJOBS={os.cpu_count()} python road_impute.py {m} {s} | tail -3
!python analyze_road.py rf all | tee f09/f1_road_rf_all.txt
!python analyze_road.py xgb rec | tee f09/f1_road_xgb_rec.txt
!python analyze_road.py rf rec LORE | tee f09/f1_road_rf_lore.txt
!python analyze_road.py xgb rec LORE | tee f09/f1_road_xgb_lore.txt

## F2. Sanity check và độ ổn định
300 cặp của fold 1. So giải thích của RF gốc (seed 0) với: RF seed 1–4 (muốn **giống**), RF học trên nhãn xáo trộn (muốn **khác**). Thêm: RF so với XGB trên cùng 2,200 cặp.

In [ ]:
run_jobs([(f'f09/sanity_{v}.json', ['python', 'sanity.py', v], f'f09/log_sanity_{v}.txt') for v in ['kg', 'seek']])
!python analyze_sanity.py kg | tee f09/f2_sanity_kg.txt
!python analyze_sanity.py seek | tee f09/f2_sanity_seek.txt

## F4. KAS với người chơi mịn hơn
Người chơi = cây con của từng con trực tiếp của aspect (độ sâu 2) + một đơn vị phần dư cho chính term aspect. Một đơn vị là người chơi khi nằm trong một aspect người chơi và chứa ít nhất một chú giải trực tiếp của một trong hai protein. Mô hình và 51 đặc trưng giữ nguyên. 300 cặp của fold 1, RF và XGB.

In [ ]:
run_jobs([(f'f09/fine_kg_{m}_f1.json', ['python', 'kas_fine.py', m, '1', '300'], f'f09/log_fine_{m}.txt') for m in ['rf', 'xgb']])
!python analyze_fine.py rf | tee f09/f4_fine_rf.txt
!python analyze_fine.py xgb | tee f09/f4_fine_xgb.txt

## Tổng hợp
Gửi Claude các file `f09/f*_*.txt` (hoặc cả thư mục `f09/`) để đối chiếu với kết quả chạy trong sandbox.

In [ ]:
!for f in f09/f*_*.txt; do echo "=== $f"; cat $f; done